# Lesson 45 · Making spectrograms — an X-ray of sound: from |STFT|² to a dB heatmap

**Goal**: turn the STFT's complex matrix into an "X-ray" you can actually read — magnitude, power and a dB heatmap.

> **Display engineering**: $10\log_{10}$ is for power, $20\log_{10}$ is for amplitude (don't mix them up); add $\varepsilon$ before the `log` to avoid log 0. Builds on reading plots in L03 and the STFT from L44.

🔗 Aurora connection: [`aurora.audio.stft.magnitude_spectrogram()`](https://github.com/djzoom/AURORA/blob/main/src/aurora/audio/stft.py) · [`aurora.audio.mel.power_to_db()`](https://github.com/djzoom/AURORA/blob/main/src/aurora/audio/mel.py)

← **Previous lesson** · [L44 · Writing the STFT](L44_stft_implement.ipynb)

> Last lesson covered **writing the STFT**: framing + windowing + FFT, checked against aurora.audio.stft.
> This lesson explores **making spectrograms**.

## This lesson's story: why does a spectrogram look like an "X-ray of music"?

A doctor uses an X-ray to look at bones — position is laid out across the film, and brightness shows density. A spectrogram does the same kind of thing: the horizontal axis is time, the vertical axis is frequency, and brightness is energy.

The STFT outputs a complex matrix `S` (shape `(n_frames, n_fft//2+1)`), and every element carries both a magnitude and a phase. What the ear perceives is not the phase but the **energy** — so we throw the phase away and keep only the magnitude:

```
mag = |S|           # magnitude spectrum, arbitrary units
pow = |S|²          # power spectrum, arbitrary units (proportional to power)
dB  = 10·log₁₀(pow + ε)  # dB spectrum, compresses the dynamic range
```

Why use dB? Sound has a dynamic range of up to 120 dB (a power ratio of 10¹²). Plot it on a linear scale and quiet sounds are nearly invisible; on a dB scale both soft and loud sounds are easy to make out — which is exactly how the ear's logarithmic perception works.

**The three coordinates for reading a spectrogram**:

| Axis | Meaning | How to compute it |
|---|---|---|
| horizontal (time) | frame m corresponds to `t = m·hop/sr` seconds | `t = np.arange(n_frames) * hop / sr` |
| vertical (frequency) | bin k corresponds to `f = k·sr/n_fft` Hz | `f = np.arange(n_fft//2+1) * sr / n_fft` |
| color (brightness) | dB power, usually normalized to -80~0 dB | `dB = 10·log₁₀(|S|² + 1e-9)` |

Task for this section: implement `plot_spectrogram(x, sr, win_len, hop)` — draw a signal as a dB magnitude spectrogram.

## Shape mnemonic (lines up with the L50 MFCC pipeline)

Fixed convention: `1 second @ 16 kHz`, `win_len=512`, `hop=256`, `center=True`:

```text
STFT complex spectrum ≈ (63, 257)    # time × frequency bin
STFT magnitude/power  ≈ (63, 257)
Mel spectrum          ≈ (63, 40)
MFCC                  ≈ (63, 13)
```

The last three lines land one lesson at a time in L46–L50; for this lesson, just remember **spectrogram = a picture of |STFT|²**.


In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine, read_wav
from aurora.audio.stft import stft, magnitude_spectrogram
from aurora.audio.mel import power_to_db

### Units on the time axis: why is (samples) / (samples/second) = seconds?

When the STFT slices frames, the **starting sample** of frame m is `m * hop_length` (frame 0 starts at sample 0, frame 1 at sample `hop`, and so on).

But the spectrogram's horizontal axis has to show **seconds**, not samples. So we convert units with the sample rate `sr`:

$$t = \frac{m \cdot \text{hop\_length}}{sr} \quad [\text{s}]$$

**Unit check** (this is the key part!):
- numerator: `m * hop_length` is measured in "samples" = "a number of samples"
- denominator: `sr` is defined as "samples per second" = $\frac{\text{samples}}{\text{s}}$
- dividing: $\frac{\text{samples}}{\text{samples}/\text{s}} = \text{samples} \times \frac{\text{s}}{\text{samples}} = \text{s}$ ✓

**A concrete example**:
- `sr = 16000` samples/second, `hop = 256` samples, frame 5 (m=5)
- starting sample: `5 * 256 = 1280` samples
- in seconds: `1280 / 16000 = 0.08` seconds
- check: `0.08 s × 16000 samples/s = 1280 samples` ✓ (converting back really does give 1280)

This kind of unit conversion is everywhere in physics. The next code cell uses this formula to compute the time-axis ticks.

### What is "power" in acoustics? Why can |S|² stand for power?

In physics, **power = work done per unit time**, measured in watts (W). For a sound wave, the intensity (power per unit area) is:

$$P_{\text{sound}} = \frac{1}{2} \cdot \rho \cdot c \cdot v^2$$

where $\rho$ is the density of the medium, $c$ is the speed of sound, and $v$ is the amplitude (of the particle velocity). The key observation: **power is proportional to the square of the amplitude**, $v^2$.

In signal processing we usually don't care about absolute physical watts (the signal was converted long ago by the analog-to-digital converter into dimensionless numbers between -1 and 1); what we care about is **relative energy**. So by convention we define:

$$\text{power spectrum} = |\text{amplitude}|^2$$

The benefits of doing it this way:
- **Energy conservation** (Parseval's theorem): the total energy is the same in the time and frequency domains
- **Consistent with physics**: it matches the physical relationship power ∝ amplitude²
- **Friendly to log scales**: $\log(A^2) = 2\log(A)$, which makes converting to dB easy

So even though the code just says `A**2`, that squaring reflects a physical principle of acoustics; it isn't an arbitrary mathematical transformation.

### Why does the STFT output complex numbers instead of giving the magnitude directly?

Every bin of the STFT is a **complex number** $S[m, k] = A[m,k] \cdot e^{i\phi[m,k]}$, made of a magnitude $A$ and a phase $\phi$.

- **Phase** is crucial in frequency analysis: it records the time delay of that frequency component. For example, two 440 Hz pure tones reinforce each other if their phases match and cancel if their phases are opposite. That matters a lot in audio processing (echo cancellation, spatial audio, and so on).
- **But what about hearing?** The ear is far less sensitive to phase than to magnitude (energy). The purpose of a spectrogram is to "see how the music's energy is distributed", so we keep the magnitude and discard the phase — that isn't waste, it's **deliberately extracting the information the ear cares about**.

Another analogy: if the STFT is a medical scan (the raw data coming out of a CT scanner), then keeping only the magnitude is like keeping the bone-density information and discarding where the organs are — good enough for some applications (like screening for osteoporosis), not enough for others (like planning surgery). A spectrogram happens to be an application that "only needs the energy".

## Concept 1: magnitude spectrum, power spectrum, dB spectrum

The STFT matrix `S` has shape `(n_frames, n_fft//2+1)`, and every element is a complex number.

- **Magnitude spectrum**: `A = np.abs(S)`, in linear amplitude units
- **Power spectrum**: `P = A**2`, proportional to energy
- **dB spectrum**: `dB = 10 * log10(P + eps)`, where eps prevents log(0)

Aurora's implementation uses eps = `1e-10` (a -100 dB noise floor) and supports a `top_db` cutoff to keep the dynamic range from getting too large.
Note: another formula common in audio, `20 * log10(A + eps)`, is equivalent to `10 * log10(A**2 + eps)` (they differ slightly only near eps).

### Why is the highest frequency the STFT can represent exactly sr/2? (a quick tour of the sampling theorem)

Imagine photographing an electric fan that's spinning fast. If the camera takes only 10 pictures a second and the fan also happens to turn exactly 10 times a second, then every time the shutter clicks the blades are in almost the same position — the photos show "a fan that's barely moving", with no hint that it's spinning at high speed. This is the same "wagon-wheel effect" you see in films and on TV: when the sampling rate of the camera can't keep up with how fast the object really changes, the eye is "fooled" into seeing a fast vibration as a slow one, or even one going backwards.

Sampling sound is the same thing. A microphone takes `sr` samples per second, which is like taking `sr` "photos" of the sound wave every second. A pure tone at frequency `f` vibrates `f` times a second; to correctly "recognize" from the samples that it's oscillating up and down (and not an almost constant straight line, or a slower fake waveform), you need to catch at least 2 samples in every cycle — one near the peak and one near the trough. That threshold of "at least 2 samples per cycle" is the core result of the **Nyquist–Shannon sampling theorem**: with sample rate `sr`, the highest frequency that can be recovered without ambiguity is

$$f_{\max} = \frac{sr}{2}$$

This is exactly the "Nyquist frequency". If the original signal contains components above `sr/2`, sampling doesn't make them vanish; instead they "disguise" themselves as some frequency below `sr/2` in the result — a phenomenon called **aliasing**, and the fan-photo illusion is the most intuitive example of it (to dig deeper, revisit L34 on aliasing). So the frequency range the STFT (and any spectral analysis of sampled data) can see has a hard upper limit of `[0, sr/2]` — that's not the result of some function parameter, it's a mathematical limit that comes from sampling itself. The only way to see higher frequencies is to raise the sample rate `sr`; adjusting `n_fft` or `hop` won't help at all.

### Choosing ε (the noise floor): why 1e-10 and 1e-8?

**The problem**: $\log(0)$ is minus infinity, so computing dB directly blows up. So we add a small ε:

$$\text{dB} = 10 \log_{10}(P + \varepsilon) \quad \text{or} \quad 20 \log_{10}(A + \varepsilon)$$

**What ε means physically**: it sets the dB value of the darkest part of the plot (where there's no signal at all), usually called the "noise floor".

**Why different ε values?**

- power form (`ε = 1e-10`): $10 \log_{10}(1e-10) = 10 \times (-10) = -100 \text{ dB}$
- amplitude form (`ε = 1e-8`): $20 \log_{10}(1e-8) = 20 \times (-8) = -160 \text{ dB}$

So the two floors are **not** the same. Recall $\log(A^2) = 2\log(A)$: an amplitude ε of 1e-8 corresponds to a power ε of $(1e-8)^2 = 1e-16$:

$$20 \log_{10}(1e-8) = 10 \log_{10}((1e-8)^2) = 10 \log_{10}(1e-16) = -160 \text{ dB}$$

So the power form with ε = 1e-10 stops at -100 dB, while the amplitude form with ε = 1e-8 goes down to -160 dB. For loud signals the two agree; they only part ways near the floor (the code below shows where).

**How to choose ε?**

Practical advice:
1. **ε too small** (e.g. 1e-15) → a very low noise floor; silent regions show speckle too, and the plot looks "dirty"
2. **ε too large** (e.g. 1e-3) → a high noise floor; soft sounds get crushed into a black blob and detail is lost
3. **A sensible range**: for 16-bit audio (a dynamic range of about 96 dB), a floor somewhere between -100 dB and -160 dB is reasonable — power ε = 1e-10 gives -100 dB, amplitude ε = 1e-8 gives -160 dB

**Try it out**: the code below compares the effect of different ε values.

### "Why 10 and 20?" — where the dB formulas come from

The dB (decibel) is a logarithmic ratio unit. It has two definitions, one for each kind of reference quantity:

**Form 1: based on power** $\text{dB}_{\text{power}} = 10 \log_{10}(P / P_{\text{ref}})$

In acoustics, the power reference is usually the intensity $P_{\text{ref}} = 10^{-12}$ W/m² (the quietest intensity the ear can hear). If we use relative energy $P = |S|^2$:

$$\text{dB}_{\text{power}} = 10 \log_{10}(|S|^2 + \varepsilon)$$

**Form 2: based on amplitude** $\text{dB}_{\text{amp}} = 20 \log_{10}(A / A_{\text{ref}})$

Because power ∝ amplitude², we have $A^2 = P$, so:

$$\text{dB}_{\text{amp}} = 20 \log_{10}(A + \varepsilon) = 20 \log_{10}(|S| + \varepsilon)$$

The derivation: $10 \log_{10}(A^2) = 10 \cdot 2 \log_{10}(A) = 20 \log_{10}(A)$

**When are the two forms equivalent?**

When $|S| \gg \varepsilon$ (the signal is far above the noise floor), the two are approximately equal. But near $|S| \approx \varepsilon$, the addition breaks the logarithm's properties:

- $10 \log_{10}(A^2 + \varepsilon_1)$ vs $20 \log_{10}(A + \varepsilon_2)$: when $A$ is very small, the $\varepsilon_1$ in the first and the $\varepsilon_2$ in the second lead to different "noise floors"

**Why not 5 or 30?**

Because the definition of the dB comes from physics (the logarithm of a power ratio); it isn't an arbitrary choice. Once you fix the reference and the power relationship, the coefficient is determined. Switch to 5 or 30 and it's no longer the standard dB, and it won't agree with measuring instruments, standard libraries and so on.

In [ ]:
# Demo: three representations of an FFT
SR = 16000
x = sine(440, duration=0.5, sample_rate=SR) + sine(880, duration=0.5, sample_rate=SR)

S = stft(x, n_fft=1024, hop_length=256)  # (n_frames, 513)
print(f"STFT shape: {S.shape}, dtype: {S.dtype}")

A = np.abs(S)                     # magnitude spectrum
P = A ** 2                        # power spectrum
dB = 10 * np.log10(P + 1e-10)    # dB spectrum

print(f"magnitude range: [{A.min():.4f}, {A.max():.4f}]")
print(f"power range:     [{P.min():.4e}, {P.max():.4e}]")
print(f"dB range:        [{dB.min():.1f}, {dB.max():.1f}] dB")

In [ ]:
# Demo: how the two dB forms agree and differ (answering "when are they approximately equal?")
import numpy as np

# Build A values from 1e-7 to 1 to show how the two forms differ
A_vals = np.logspace(-7, 0, 50)  # from 1e-7 to 1

# Method 1: amplitude form, ε = 1e-8
dB_amp = 20 * np.log10(A_vals + 1e-8)

# Method 2: power form, ε = 1e-10
P_vals = A_vals ** 2
dB_pow = 10 * np.log10(P_vals + 1e-10)

# The difference
diff = np.abs(dB_amp - dB_pow)

# Plot it
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: the two curves compared
ax1.semilogx(A_vals, dB_amp, label="20·log₁₀(A + 1e-8) [amplitude]", linewidth=2)
ax1.semilogx(A_vals, dB_pow, label="10·log₁₀(A² + 1e-10) [power]", linewidth=2, linestyle="--")
ax1.axvline(1e-8, color='r', linestyle=':', alpha=0.5, label="A=1e-8 marker")
ax1.set_xlabel("magnitude A (log scale)")
ax1.set_ylabel("dB value")
ax1.set_title("The two dB formulas compared")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: size of the difference
ax2.loglog(A_vals, diff, linewidth=2, color='purple')
ax2.axhline(1, color='orange', linestyle='--', label="difference = 1 dB")
ax2.set_xlabel("magnitude A (log scale)")
ax2.set_ylabel("difference |dB₁ - dB₂| (dB)")
ax2.set_title("How the difference changes with A")
ax2.legend()
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

# Table of values at key points
print("\nThe two forms compared at key magnitudes:")
print("=" * 70)
print(f"{'A magnitude':<12} {'20·log(A+1e-8)':<20} {'10·log(A²+1e-10)':<20} {'diff':<10}")
print("=" * 70)
for A in [1, 0.1, 0.01, 1e-4, 1e-6, 1e-8]:
    db1 = 20 * np.log10(A + 1e-8)
    db2 = 10 * np.log10(A**2 + 1e-10)
    diff_val = abs(db1 - db2)
    print(f"{A:<12.0e} {db1:<20.2f} {db2:<20.2f} {diff_val:<10.3f}")

print("\nConclusion:")
print("  • for A ≥ 0.01 (strong signals): the two are almost identical (difference < 0.1 dB)")
print("  • around A ≈ 1e-5 (weak signals): they start to differ noticeably (≈ 3 dB)")
print("  • for A ≤ 1e-6 (very weak signals): the power form is pinned at its -100 dB floor while the amplitude form keeps falling, so the gap grows to tens of dB")
print("  • in practice, for A ≥ 1e-4 you can treat them as 'equivalent'")

In [ ]:
# Demo: how different ε values change the noise floor of the dB spectrum
SR = 16000
x = sine(440, duration=0.5, sample_rate=SR) + sine(880, duration=0.5, sample_rate=SR)

S = stft(x, n_fft=1024, hop_length=256)
A = np.abs(S)

# dB spectra for different ε (amplitude form)
eps_list = [1e-15, 1e-10, 1e-8, 1e-5]
dB_list = [20 * np.log10(A + eps) for eps in eps_list]

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
axes = axes.flatten()

for ax, dB, eps in zip(axes, dB_list, eps_list):
    im = ax.imshow(dB.T, origin="lower", aspect="auto", cmap="inferno")
    noise_floor_dB = 20 * np.log10(eps)
    ax.set_title(f"ε = {eps:.0e}\n(noise floor = {noise_floor_dB:.0f} dB)")
    ax.set_ylabel("frequency (Hz)")
    fig.colorbar(im, ax=ax)

axes[0].set_ylabel("frequency (Hz)")
axes[2].set_ylabel("frequency (Hz)")
plt.tight_layout()
plt.show()

print("What to look for:")
print(f"  ε=1e-15: noise floor = -300 dB, speckle in the silent regions (too much detail)")
print(f"  ε=1e-10: noise floor = -200 dB, still speckly in the quietest regions")
print(f"  ε=1e-8:  noise floor = -160 dB, clean and usable (recommended)")
print(f"  ε=1e-5:  noise floor = -100 dB, soft sounds look slightly compressed")
print("\nIn this amplitude form, ε=1e-8 (-160 dB) is the recommended value; the power form's ε=1e-10 gives a -100 dB floor instead.")

### Deriving the STFT frame count: why `1 + (N - win_len) // hop`?

When you take the STFT of a signal of length N, you need to work out how many frames fit (each frame holds `win_len` samples, and neighboring frames are `hop` samples apart).

**Derivation**:
1. frame 0 starts at sample 0
2. frame 1 starts at sample hop
3. frame m starts at sample `m·hop`

**The constraint on the last frame**: the last frame has to cover a full `win_len` samples, so its **starting position** can be no later than `N - win_len` (so that it can still read sample `N-1`).

So the index of the last frame, `m_max`, satisfies:
$$m_{\max} \cdot \text{hop} \leq N - \text{win\_len}$$

$$m_{\max} \leq \frac{N - \text{win\_len}}{\text{hop}}$$

Since `m_max` must be an integer, round down:
$$m_{\max} = \left\lfloor \frac{N - \text{win\_len}}{\text{hop}} \right\rfloor = (N - \text{win\_len}) \text{ // } \text{hop}$$

**Total number of frames** = largest frame index + 1 (because frame 0 counts too):
$$n_{\text{frames}} = m_{\max} + 1 = 1 + \left\lfloor \frac{N - \text{win\_len}}{\text{hop}} \right\rfloor = 1 + (N - \text{win\_len}) \text{ // } \text{hop}$$

**A concrete example**:
- `N = 8000` (1 second @ 8 kHz), `win_len = 256`, `hop = 128`
- $N - \text{win\_len} = 8000 - 256 = 7744$
- $(7744) \text{ // } 128 = 60$
- $n_{\text{frames}} = 1 + 60 = 61$ frames
- check: the last frame starts at sample $60 \times 128 = 7680$ and ends at sample $7680 + 256 - 1 = 7935 < 8000$ ✓

**Note**: this formula assumes the window lies entirely inside the signal (no padding). With `center=True` (which the L44 STFT does), the actual frame count can differ somewhat, but the principle is the same.

## Concept 2: the key parameters of `imshow`

By default `plt.imshow` draws row 0 of the matrix at the top (image coordinates), but frequency plots conventionally put low frequencies at the bottom, so you must set:

```
origin="lower"   # row 0 (DC / 0 Hz) at the bottom
aspect="auto"    # don't force square pixels; fit any time–frequency proportions
extent=[t_min, t_max, f_min, f_max]  # make the axes show real units
```

The order of `extent` is `[x_left, x_right, y_bottom, y_top]`, which here is
`[0, duration_s, 0, sr/2]`.
`sr/2` is the Nyquist frequency, the highest frequency the STFT can represent.

### How imshow's coordinates map to the transpose (the intuition)

The STFT output matrix `S` has shape `(n_frames, n_fft//2+1)`, where:
- **rows (axis 0)** = time (frame index)
- **columns (axis 1)** = frequency (frequency bin)

But by default `plt.imshow` does this:
- **rows (axis 0)** → **vertical axis (y axis)**
- **columns (axis 1)** → **horizontal axis (x axis)**

**What happens if you don't transpose?**

```
STFT matrix (63, 257)
     rows=time, columns=frequency
          ↓ imshow
     vertical=time (up/down), horizontal=frequency (left/right)
     ❌ That's backwards! We want frequency on the vertical axis!
```

**After transposing with `.T`**:

```
transposed matrix (257, 63)
     rows=frequency, columns=time
          ↓ imshow
     vertical=frequency (up/down), horizontal=time (left/right)
     ✓ Correct!
```

**What the origin parameter does**:

`imshow` uses **image coordinates** by default (row 0 at the top), but frequency plots conventionally use **Cartesian coordinates** (y=0 at the bottom).

After transposing, row 0 of the matrix stands for 0 Hz (DC, the lowest frequency). To put low frequencies at the bottom of the plot (as intuition expects), you must use `origin="lower"`.

```
origin="upper" (default)        origin="lower" (after setting it)
   row 0: 0 Hz ────────────       row 256: 8000 Hz ────
   row 1: 31.25 Hz                row 255: 7968.75 Hz
   ...                             ...
   row 256: 8000 Hz ════════       row 0: 0 Hz ════════
   ↑                               ↑
   low frequencies on top (odd!)   low frequencies at the bottom (as expected)
```

**The full effect, together with extent**:

```python
plt.imshow(S.T, origin="lower", aspect="auto",
           extent=[0, duration, 0, sr/2])
```

- `.T`: puts frequency on the vertical axis
- `origin="lower"`: puts 0 Hz at the bottom
- `extent=[0, duration, 0, sr/2]`: labels the axes in real seconds and hertz


### ✨ The extent parameter in detail: why this order?

`extent=[x_left, x_right, y_bottom, y_top]` looks odd because the order differs from the familiar `[x_min, x_max, y_min, y_max]`.

**matplotlib's documented convention** (not NumPy's — this is specific to imshow):

- `extent[0]` = left edge of the x axis (smallest time) = 0
- `extent[1]` = right edge of the x axis (largest time) = total signal duration duration_s
- `extent[2]` = bottom edge of the y axis (smallest frequency, because origin='lower') = 0 Hz
- `extent[3]` = top edge of the y axis (largest frequency) = sr/2 (the Nyquist frequency)

**Why is the y axis "bottom→top" rather than "top→bottom"?**

Because we set `origin='lower'`, imshow flips the direction of the y axis. So when you give the coordinate range, you use the "bottom→top" order to match the actual direction of the axis.

**Where the specific values come from**:
- `duration = len(x) / sr`: total number of samples divided by the sample rate = seconds
- `sr / 2`: the Nyquist frequency. The STFT uses a real-valued FFT and only outputs frequencies from 0 to sr/2 (frequencies above sr/2 can't be represented unambiguously at sample rate sr)

**How it relates to the matrix shape**:
- the matrix shape is `(n_frames, n_fft//2+1)` (before transposing)
- `n_frames` isn't used directly; the duration comes from `duration` and the sample rate
- `n_fft//2+1 = n_bins` isn't used directly either; the top edge comes from `sr/2`

The benefit: even if you change `hop` or `n_fft`, as long as the signal length and sample rate stay the same, `extent` stays the same and the plot rescales automatically.

### The trade-off between frequency and time resolution (in concrete numbers)

**Frequency resolution** = the smallest frequency spacing the spectrum can resolve, defined as:
$$\Delta f = \frac{sr}{n\_fft} \quad [\text{Hz/bin}]$$

**Time resolution** = the smallest time spacing on the time axis, defined as:
$$\Delta t = \frac{\text{hop\_length}}{sr} \quad [\text{s/frame}]$$

**Concrete examples** (sr=8000 Hz):

| Setup | win_len | hop | n_fft | Δf (Hz/bin) | Δt (ms/frame) | Use |
|------|---------|-----|-------|-------------|-----------|------|
| high time resolution | 256 | 64 | 256 | **31.25** | **8** | detecting fast events (plucked strings, keystrokes) |
| balanced | 512 | 256 | 512 | **15.63** | **32** | general audio analysis |
| high frequency resolution | 2048 | 512 | 2048 | **3.91** | **64** | telling close pitches apart precisely (440Hz vs 445Hz) |

**Working out whether two tones can be resolved**:

Two pure tones at 440 Hz and 445 Hz. How narrow do the bins need to be to separate them?

- **Δf = 31.25 Hz**: both frequencies fall in the same bin (bin14 = 437.5 Hz, bin15 = 468.75 Hz) → **can't be resolved**
- **Δf = 15.63 Hz**: 440 Hz and 445 Hz both fall nearest bin28 (437.5 Hz; bin29 is 453.125 Hz) → **still can't be resolved**
- **Δf = 3.91 Hz**: 440 Hz falls in bin113 (441.4 Hz), 445 Hz in bin114 (445.3 Hz) → **resolved** (in neighboring bins)

**Why does a small Δf help?**

$$n_{\text{fft}} = \Delta f_{\text{desired}}^{-1} \times sr$$

To get Δf = 3.91 Hz you need `n_fft = 8000 / 3.91 ≈ 2048`.

But the bigger `n_fft`, the more time a single frame covers: frame length = `n_fft / sr = 2048 / 8000 = 0.256` seconds, and time resolution drops accordingly.

That's exactly the **time–frequency trade-off** summarized in the table above.

In [ ]:
# Demo: how origin and aspect affect the plot
SR = 16000
dur = 0.5
x = sine(440, duration=dur, sample_rate=SR)
S = stft(x, n_fft=1024, hop_length=256)
A = np.abs(S).T   # transpose: (frequency bins, frames) → imshow rows=frequency, columns=time

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].imshow(A, aspect="auto")
axes[0].set_title("origin=upper (default) — frequency upside down")

axes[1].imshow(A, origin="lower", aspect="auto",
               extent=[0, dur, 0, SR/2])
axes[1].set_title("origin=lower + extent — correct")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("frequency (Hz)")

plt.tight_layout()
plt.show()

## Concept 3: computing the time and frequency axes exactly

**Frequency axis**: rfft outputs `n_fft//2 + 1` bins, and bin `k` corresponds to the frequency:

```
f[k] = k * sr / n_fft      k = 0, 1, ..., n_fft//2
```

That is, `f = np.arange(n_fft//2 + 1) * sr / n_fft`.

**Time axis**: the center sample of frame `t` sits at `t * hop_length` (already aligned when `center=True` is on);
in seconds:

```
t[i] = i * hop_length / sr     i = 0, 1, ..., n_frames-1
```

`extent` only takes the boundaries, so `[0, n_frames*hop/sr, 0, (n_fft//2)*sr/n_fft]` is accurate enough.

In [ ]:
# Demo: computing the axis ticks by hand
SR = 16000
N_FFT = 1024
HOP = 256

f_bins = np.arange(N_FFT // 2 + 1) * SR / N_FFT   # 513 frequencies
print(f"number of frequency bins: {len(f_bins)}")
print(f"frequency resolution: {SR/N_FFT:.2f} Hz/bin")
print(f"highest frequency: {f_bins[-1]:.0f} Hz  (= SR/2 = {SR//2} Hz)")

x = sine(440, duration=1.0, sample_rate=SR)
S = stft(x, n_fft=N_FFT, hop_length=HOP)
n_frames = S.shape[0]
t_bins = np.arange(n_frames) * HOP / SR
print(f"\nnumber of frames: {n_frames}")
print(f"time resolution: {HOP/SR*1000:.1f} ms/frame")
print(f"total duration covered: {t_bins[-1]:.3f} s")

## 1. ✏️ Implement `plot_spectrogram(x, sr, win_len=1024, hop=256)`

**A five-step recipe**:

| Step | Code | Notes |
|---|---|---|
| 1 | `S = my_stft(x, win_len, hop)` or `stft(x,...)` | get the complex STFT matrix |
| 2 | `power = np.abs(S) ** 2` | power spectrum |
| 3 | `db = 10 * np.log10(power + 1e-9)` | dB spectrum (add ε to avoid log(0)) |
| 4 | `plt.imshow(db.T, origin='lower', aspect='auto', ...)` | `.T`: frequency on the vertical axis; `origin='lower'`: low frequencies at the bottom |
| 5 | compute the time/frequency axes, set xlabel/ylabel | `t = np.arange(n_frames)*hop/sr`, `f = k*sr/win_len` |

**Acceptance criteria**:
- a two-tone 440 Hz + 880 Hz signal → two clear horizontal lines
- `db.shape == (n_frames, win_len//2+1)` (before transposing)
- the function returns the `db` array (for the numerical check)

In [ ]:
def plot_spectrogram(x, sr, win_len=1024, hop=256):
    """Draw a signal as a dB magnitude spectrogram. Returns the dB matrix for numerical checking.

    Parameters
    ----------
    x       : 1-D float array, the time-domain signal
    sr      : sample rate (Hz)
    win_len : FFT window length (samples)
    hop     : hop length (samples)

    Returns
    -------
    dB : ndarray, shape (n_fft//2+1, n_frames)
    """
    raise NotImplementedError("Complete TODO steps 1–3, then delete this line")  # ← delete this line before filling in the implementation

    # ✏️ TODO step 1: call stft() to get the complex matrix S, shape (n_frames, win_len//2+1)
    S = ...

    # ✏️ TODO step 2: np.abs(S).T gives the (frequency, time) magnitude matrix; then convert to dB
    # Note: this uses 20*log10(A + 1e-8) (amplitude form), which is approximately equivalent to
    # Aurora power_to_db's 10*log10(P + 1e-10) (power form), but the eps floors differ (see the ε section above)
    A = ...
    dB = ...

    dur = len(x) / sr

    # ✏️ TODO step 3: imshow + colorbar + axis labels (origin="lower", aspect="auto")
    fig, ax = plt.subplots(figsize=(10, 4))
    ...

    return dB  # return the dB matrix for checking outside


In [ ]:
# Visual check: are the two horizontal lines clear?
SR = 16000
x_test = sine(440, duration=1.0, sample_rate=SR) + sine(880, duration=1.0, sample_rate=SR)

try:
    student_dB = plot_spectrogram(x_test, SR)
    if student_dB is not None and hasattr(student_dB, "shape"):
        assert student_dB.shape[0] == 513, (
            f"there should be 513 frequency bins, got {student_dB.shape[0]}"
        )
        assert student_dB.max() > -20, (
            f"max dB value {student_dB.max():.1f} dB is too low: check the magnitude computation"
        )
        print(f"✅ plot_spectrogram passes: shape={student_dB.shape}, "
              f"dB.max()={student_dB.max():.1f} dB")
    else:
        print("⚠️  plot_spectrogram returned None: make sure the function ends with return dB")
except (NotImplementedError, TypeError):
    print("⬜ Not implemented yet: delete the raise NotImplementedError line and fill in the TODOs")

# Numerical check (reference implementation, also used for the color range in the parameter experiment below)
# The alias _stft: this is only to tell this stft apart from the stft() imported at the top
# of the lesson, which you call inside plot_spectrogram — they are actually the same function.
# The alias is purely so that this "reference answer" code doesn't share a name with your own
# implementation, making the comparison easier to follow; it is not a different implementation.
from aurora.audio.stft import stft as _stft
S_ref = _stft(x_test, n_fft=1024, hop_length=256)
A_ref = np.abs(S_ref).T
dB_ref = 20 * np.log10(A_ref + 1e-8)

assert dB_ref.max() > -20, "the bright lines' dB value is too low: check the magnitude computation"
assert dB_ref.shape[0] == 513, f"there should be 513 frequency bins, got {dB_ref.shape[0]}"
print(f"✅ reference implementation: shape={dB_ref.shape}, dB_ref.max()={dB_ref.max():.1f} dB")
print("   (there should be clear bright horizontal lines at 440 Hz and 880 Hz)")


## Parameter experiment: change win_len and hop to see the time–frequency trade-off

`win_len` (the window length) and `hop` (the hop length) control the **time–frequency resolution trade-off**:

| Parameter | When you change it | The cost |
|------|--------|------|
| `win_len` up | frequency resolution improves (narrower bins) | time resolution drops (each frame covers more time) |
| `hop` down | time resolution improves (denser frames) | more computation |

**What to expect**:
- `win_len=256, hop=64`: the horizontal lines get thick (blurry frequency), the time axis is very dense
- `win_len=2048, hop=512`: the horizontal lines are very thin (sharp frequency), transient events blur

In [ ]:
SR = 16000
x_exp = sine(440, duration=1.0, sample_rate=SR) + sine(880, duration=1.0, sample_rate=SR)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    (256,  64,  "win=256 hop=64\n(sharp in time, blurry in frequency)"),
    (1024, 256, "win=1024 hop=256\n(balanced, the default)"),
    (2048, 512, "win=2048 hop=512\n(sharp in frequency, blurry in time)"),
]

for ax, (win, hop, title) in zip(axes, configs):
    S = stft(x_exp, n_fft=win, hop_length=hop)
    A = np.abs(S).T
    dB = 20 * np.log10(A + 1e-8)
    dur = len(x_exp) / SR
    im = ax.imshow(dB, origin="lower", aspect="auto",
                   extent=[0, dur, 0, SR/2], cmap="inferno",
                   vmin=dB_ref.max()-60, vmax=dB_ref.max())
    ax.set_title(title)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("frequency (Hz)")
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()
print("✅ Watch how the width of the bright lines changes across the three plots")

## Wrap-up

This lesson implemented `plot_spectrogram()`, which outputs a time–frequency heatmap in dB.
The core chain of operations is `stft → np.abs → square/log → imshow`, where `origin="lower"` and `extent` decide whether the time and frequency axes line up.
The processing you write here maps directly onto Aurora's `aurora.audio.stft.magnitude_spectrogram()` and `aurora.audio.mel.power_to_db()`. The next lesson (L46) carries on from this spectrogram and builds the mel filterbank.

## Aurora connection check — magnitude_spectrogram and power_to_db

Aurora's packaged `magnitude_spectrogram()` and `power_to_db()` are equivalent to the chain you wrote by hand in this lesson, but they use the power form:

| Form | Formula | eps | dB floor |
|------|------|-----|-------|
| amplitude form (this lesson's exercise) | `20·log10(A + 1e-8)` | 1e-8 | ≈ −160 dB |
| power form (Aurora API) | `10·log10(P + 1e-10)` | 1e-10 | ≈ −100 dB |

For signals with A≥0.01 the two are numerically about equal; only near silence (A≈0) do their noise floors differ.
The code below checks that on the 440+880 Hz test signal their largest difference is < 1 dB.

In [ ]:
# Aurora connection: check that magnitude_spectrogram / power_to_db agree with the hand-written chain
SR = 16000
x_verify = sine(440, duration=0.5, sample_rate=SR) + sine(880, duration=0.5, sample_rate=SR)

# Hand-written chain (amplitude form, this lesson's exercise formula)
S_hand = stft(x_verify, n_fft=1024, hop_length=256)
A_hand = np.abs(S_hand).T
dB_hand = 20 * np.log10(A_hand + 1e-8)

# Aurora API (power form)
dB_aurora = power_to_db(np.abs(S_hand).T ** 2)

# Compare only in the strong-signal region where neither has been flattened: power_to_db's top_db=80
# clips the floor at (peak - 80) dB ≈ -32 dB, so the comparison threshold is -30 dB to stay clear of it
mask = dB_hand > -30
diff = np.abs(dB_hand[mask] - dB_aurora[mask])
print(f"hand-written chain dB range: [{dB_hand.min():.1f}, {dB_hand.max():.1f}] dB")
print(f"Aurora dB range:             [{dB_aurora.min():.1f}, {dB_aurora.max():.1f}] dB")
print(f"largest difference in the signal region: {diff.max():.2f} dB (< 1 dB counts as equivalent)")
assert diff.max() < 1.0, f"difference too large: {diff.max():.2f} dB"
print("✅ The Aurora API and the hand-written chain agree in the strong-signal region (the top_db floor and eps only affect weak signals / silence)")

# Also demo calling magnitude_spectrogram directly (note: it returns the linear |STFT|, not dB)
mag_direct = magnitude_spectrogram(x_verify)
print(f"magnitude_spectrogram() output shape: {mag_direct.shape} (linear magnitude)")
print("✅ magnitude_spectrogram gives |STFT|; wrap it in power_to_db and you have this lesson's dB chain")

## ✏️ Whiteboard challenge: a spectrogram by hand (aim for 10 minutes)

Cover the screen and answer on paper:

**Question 1**: `sr=8000, win_len=256, hop=128, duration=1.0s`:
- total number of samples N = sr × duration = ?
- n_frames = 1 + (N - win_len) // hop = ?
- spectrogram output shape (before transposing) = ?

**Question 2**: what time (seconds) and frequency (Hz) does `S[5, 10]` correspond to?
(`sr=8000, win_len=256, hop=128`)

**Question 3**: why does `plt.imshow` need `.T` (the transpose) and `origin='lower'`?
(Hint: in the STFT matrix rows=time, columns=frequency; where does imshow put the vertical origin by default?)

**Question 4**: what does the ε (e.g. 1e-9) in `dB = 10·log₁₀(|S|² + ε)` do?
What happens when `|S|=0` if you don't add ε?

**Question 5**: in a spectrogram with `win_len=256, sr=8000`, which bin does 440 Hz show up in?
(`k = round(f / Δf) = round(440 * 256 / 8000) = ?`)

Once you've worked it out, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: shape calculation
sr1, win1, hop1, dur1 = 8000, 256, 128, 1.0
N1 = int(sr1 * dur1)
n_frames1 = 1 + (N1 - win1) // hop1
n_bins1 = win1 // 2 + 1
assert N1 == 8000
assert n_frames1 == 61
assert n_bins1 == 129
print(f"Q1 ✅  N={N1}, n_frames={n_frames1}, spectrogram shape (before transposing)=({n_frames1},{n_bins1})")

# Q2: the time and Hz for a frame/bin
m, k = 5, 10
t_sec = m * hop1 / sr1
f_hz  = k * sr1 / win1
assert abs(t_sec - 0.08) < 1e-10
assert f_hz == 312.5
print(f"Q2 ✅  S[5,10] → t={t_sec:.3f}s, f={f_hz:.1f} Hz")

# Q3: imshow transpose and origin
# Check: imshow defaults to origin='upper' (row 0 at the top); a spectrogram needs low frequencies at the bottom
test_mat = np.arange(6).reshape(2, 3)  # rows=time (2 frames), columns=frequency (3 bins)
# After transposing: rows=frequency (3), columns=time (2); origin='lower' puts row0 (low frequency) at the bottom
assert test_mat.T.shape == (3, 2)  # after transposing, rows=frequency
print(f"Q3 ✅  .T makes rows=frequency (imshow draws by rows); origin='lower' puts row0 (low frequency) at the bottom (the default upper → low frequencies on top)")

# Q4: ε prevents log(0)
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    log_zero = np.log10(0.0)
assert np.isinf(log_zero)  # log(0) = -inf
log_eps = 10 * np.log10(0.0 + 1e-9)
assert np.isfinite(log_eps) and log_eps < 0
print(f"Q4 ✅  log₁₀(0)={log_zero} (−inf); adding ε → {log_eps:.1f} dB (a finite value, standing for a very weak signal)")

# Q5: the bin for 440 Hz
f_target, win5, sr5 = 440, 256, 8000
k_bin = round(f_target * win5 / sr5)
f_actual = k_bin * sr5 / win5
assert k_bin == 14
assert f_actual == 437.5
print(f"Q5 ✅  440 Hz → bin k={k_bin} (exact frequency={f_actual} Hz, Δf={sr5/win5:.1f} Hz/bin)")
print("\n🎉 Spectrogram whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l45_review = {
    "db_formula":              None,  # remember dB=10·log₁₀(|S|²+ε)? True/False
    "plot_spectrogram_impl":   None,  # plot_spectrogram implemented, both lines clearly visible? True/False
    "transpose_and_origin":    None,  # understand why .T and origin='lower' are needed? True/False
    "time_freq_axis_calc":     None,  # can compute t=m·hop/sr and f=k·sr/win by hand? True/False
    "whiteboard_passed":       None,  # whiteboard challenge done on paper? True/False
    "shape_memo_known":        None,  # remember the (63,257)→(63,40)→(63,13) mnemonic? True/False
}

unfilled = [k for k, v in l45_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l45_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L45 fully cleared! On to L46: the mel scale')

---

→ **Next lesson** · [L46 · The mel scale](L46_mel.ipynb)

> Next lesson covers **the mel scale**: the ear's logarithmic pitch perception, mel = 2595·log₁₀(1+f/700), and triangular filters.